In [1]:
!pip install -q transformers peft accelerate datasets huggingface_hub wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.6 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import wandb
import gc
from torch.optim import AdamW
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset
from huggingface_hub import login

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ==========================================
# AUTHENTICATION
# ==========================================
HF_TOKEN = "your_hf_token"
WANDB_KEY = "Your_wandb_key"

login(token=HF_TOKEN)
wandb.login(key=WANDB_KEY)

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_NAME = 'Qwen/Qwen3-0.6B'
PARAGRAPH_DIM = 1024
PREFIX_LEN = 32
MAX_TEXT_LEN = 128
BATCH_SIZE = 4
LEARNING_RATE = 5e-5
SUBSET_SIZE = 5000      

LAMBDA_VALUES = [0.5, 1.0, 2.0] 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

Device: cuda


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: subhavpathak18 (subhavpathak18-indian-institute-of-information-technolog) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [3]:
class SingleTokenMLP(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(paragraph_dim, 2048), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(2048, decoder_hidden_dim)
        )
    def forward(self, x):
        return self.net(x)

class EndToEndInverter(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim, prefix_len, decoder_model):
        super().__init__()
        self.prefix_len = prefix_len
        self.decoder_hidden_dim = decoder_hidden_dim
        self.m_parallel_mlps = nn.ModuleList([
            SingleTokenMLP(paragraph_dim, decoder_hidden_dim) for _ in range(prefix_len)
        ])
        self.decoder = decoder_model

    def forward(self, paragraph_embs, text_input_ids, attention_mask):
        device = paragraph_embs.device
        batch_size = paragraph_embs.shape[0]

        outputs = [mlp(paragraph_embs) for mlp in self.m_parallel_mlps]
        prefix_embeds = torch.stack(outputs, dim=1)

        text_embeds = self.decoder.get_input_embeddings()(text_input_ids)
        prefix_embeds = prefix_embeds.to(dtype=text_embeds.dtype)
        inputs_embeds = torch.cat([prefix_embeds, text_embeds], dim=1)

        prefix_labels = torch.full((batch_size, self.prefix_len), -100, dtype=torch.long, device=device)
        labels = torch.cat([prefix_labels, text_input_ids], dim=1)

        prefix_mask = torch.ones((batch_size, self.prefix_len), dtype=attention_mask.dtype, device=device)
        concat_mask = torch.cat([prefix_mask, attention_mask], dim=1)

        out = self.decoder(inputs_embeds=inputs_embeds, attention_mask=concat_mask, labels=labels)
        ce_loss = out.loss

        mean_prefix = prefix_embeds.mean(dim=1)
        mask_exp = attention_mask.unsqueeze(-1).to(dtype=text_embeds.dtype)
        mean_text = (text_embeds * mask_exp).sum(dim=1) / mask_exp.sum(dim=1).clamp(min=1e-9)
        target = torch.ones(batch_size, device=device)
        aux_loss = F.cosine_embedding_loss(mean_prefix.float(), mean_text.float(), target)

        # RETURN BOTH LOSSES SEPARATELY so we can combine them with any lambda!
        return ce_loss, aux_loss

def prepare_batch(batch_samples):
    paragraph_embs_list, input_ids_list, attention_mask_list = [], [], []
    for sample in batch_samples:
        sent_emb = torch.tensor(sample["sentence_embeddings"], dtype=torch.float32)
        paragraph_embs_list.append(sent_emb)

        ids = sample["input_ids"]
        seq_len = min(len(ids), MAX_TEXT_LEN)

        padded_ids = np.full(MAX_TEXT_LEN, tokenizer.pad_token_id, dtype=np.int64)
        padded_ids[:seq_len] = ids[:seq_len]
        input_ids_list.append(torch.tensor(padded_ids))

        mask = np.zeros(MAX_TEXT_LEN, dtype=np.float32)
        mask[:seq_len] = 1.0
        attention_mask_list.append(torch.tensor(mask))

    return torch.stack(paragraph_embs_list), torch.stack(input_ids_list), torch.stack(attention_mask_list)

In [4]:
# # ==========================================
# # THE SWEEP: 3 EXPERIMENTS, ONE AFTER ANOTHER
# # ==========================================

# results_summary = {}

# for lam in LAMBDA_VALUES:
#     print(f"\n{'='*80}")
#     print(f"  EXPERIMENT: lambda = {lam}")
#     print(f"{'='*80}\n")

#     # --- 1. Start a fresh wandb run for this lambda ---
#     run = wandb.init(
#         project="qwen-lambda-sweep",
#         name=f"lambda={lam}",
#         config={"lambda": lam, "lr": LEARNING_RATE, "batch_size": BATCH_SIZE, "subset": SUBSET_SIZE},
#         reinit=True
#     )

#     # --- 2. Build a FRESH model from scratch (fair comparison!) ---
#     base_decoder = AutoModelForCausalLM.from_pretrained(
#         MODEL_NAME, torch_dtype=torch.bfloat16, device_map={"": device}
#     )

#     lora_config = LoraConfig(
#         task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
#         target_modules=["q_proj", "v_proj"]
#     )
#     decoder_with_lora = get_peft_model(base_decoder, lora_config)

#     model = EndToEndInverter(
#         paragraph_dim=PARAGRAPH_DIM,
#         decoder_hidden_dim=base_decoder.config.hidden_size,
#         prefix_len=PREFIX_LEN,
#         decoder_model=decoder_with_lora
#     )
#     model = model.to(device)
#     model = nn.DataParallel(model)
#     model.train()

#     optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

#     # --- 3. Stream 5,000 samples and train ---
#     stream = load_dataset(
#         "jg-eno/msmarco-v5.1-Qwen-Embeddings", split="train", streaming=True
#     ).shuffle(seed=42)

#     batch_buffer = []
#     total_loss = 0
#     batch_count = 0
#     sample_count = 0

#     progress_bar = tqdm(stream, desc=f"λ={lam}", total=SUBSET_SIZE)

#     for sample in progress_bar:
#         batch_buffer.append(sample)
#         sample_count += 1

#         if len(batch_buffer) == BATCH_SIZE:
#             paragraph_embs, text_input_ids, attention_mask = prepare_batch(batch_buffer)
#             paragraph_embs = paragraph_embs.to(device)
#             text_input_ids = text_input_ids.to(device)
#             attention_mask = attention_mask.to(device)

#             optimizer.zero_grad()

#             ce_loss, aux_loss = model(paragraph_embs, text_input_ids, attention_mask)
#             ce_loss = ce_loss.mean()     # Average across GPUs
#             aux_loss = aux_loss.mean()   # Average across GPUs

#             # THIS IS WHERE LAMBDA MATTERS!
#             total_combined_loss = ce_loss + lam * aux_loss

#             total_combined_loss.backward()
#             optimizer.step()

#             total_loss += total_combined_loss.item()
#             batch_count += 1
#             batch_buffer = []

#             # Log ALL three values to wandb for comparison
#             wandb.log({
#                 "total_loss": total_combined_loss.item(),
#                 "ce_loss": ce_loss.item(),
#                 "aux_loss": aux_loss.item(),
#                 "batch": batch_count
#             })

#             progress_bar.set_postfix({
#                 "Total": f"{total_combined_loss.item():.4f}",
#                 "CE": f"{ce_loss.item():.4f}",
#                 "Aux": f"{aux_loss.item():.4f}"
#             })

#         # Stop after 5000 samples
#         if sample_count >= SUBSET_SIZE:
#             break

#     avg_loss = total_loss / max(batch_count, 1)
#     results_summary[lam] = avg_loss
#     print(f"\nλ={lam} finished! Average Loss: {avg_loss:.4f}")

#     wandb.finish()

#     # --- 4. Clean up GPU memory before next experiment ---
#     del model, optimizer, decoder_with_lora, base_decoder
#     torch.cuda.empty_cache()
#     gc.collect()
#     print("GPU memory cleared for next experiment.\n")

# # ==========================================
# # FINAL SUMMARY TABLE
# # ==========================================
# print("\n" + "=" * 50)
# print("  LAMBDA SWEEP RESULTS")
# print("=" * 50)
# for lam, avg in results_summary.items():
#     print(f"  λ = {lam}  →  Average Loss = {avg:.4f}")

# best_lambda = min(results_summary, key=results_summary.get)
# print(f"\n  🏆 BEST LAMBDA = {best_lambda}")
# print("=" * 50)

In [5]:
# # ==========================================
# # EXTRA EXPERIMENT: lambda = 0.25
# # ==========================================
# lam = 0.25

# print(f"\n{'='*80}")
# print(f"  EXPERIMENT: lambda = {lam}")
# print(f"{'='*80}\n")

# run = wandb.init(
#     project="qwen-lambda-sweep",
#     name=f"lambda={lam}",
#     config={"lambda": lam, "lr": LEARNING_RATE, "batch_size": BATCH_SIZE, "subset": SUBSET_SIZE},
#     reinit=True
# )

# base_decoder = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME, torch_dtype=torch.bfloat16, device_map={"": device}
# )

# lora_config = LoraConfig(
#     task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
#     target_modules=["q_proj", "v_proj"]
# )
# decoder_with_lora = get_peft_model(base_decoder, lora_config)

# model = EndToEndInverter(
#     paragraph_dim=PARAGRAPH_DIM,
#     decoder_hidden_dim=base_decoder.config.hidden_size,
#     prefix_len=PREFIX_LEN,
#     decoder_model=decoder_with_lora
# )
# model = model.to(device)
# model = nn.DataParallel(model)
# model.train()

# optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

# stream = load_dataset(
#     "jg-eno/msmarco-v5.1-Qwen-Embeddings", split="train", streaming=True
# ).shuffle(seed=42)

# batch_buffer = []
# total_loss = 0
# batch_count = 0
# sample_count = 0

# progress_bar = tqdm(stream, desc=f"λ={lam}", total=SUBSET_SIZE)

# for sample in progress_bar:
#     batch_buffer.append(sample)
#     sample_count += 1

#     if len(batch_buffer) == BATCH_SIZE:
#         paragraph_embs, text_input_ids, attention_mask = prepare_batch(batch_buffer)
#         paragraph_embs = paragraph_embs.to(device)
#         text_input_ids = text_input_ids.to(device)
#         attention_mask = attention_mask.to(device)

#         optimizer.zero_grad()

#         ce_loss, aux_loss = model(paragraph_embs, text_input_ids, attention_mask)
#         ce_loss = ce_loss.mean()
#         aux_loss = aux_loss.mean()

#         total_combined_loss = ce_loss + lam * aux_loss

#         total_combined_loss.backward()
#         optimizer.step()

#         total_loss += total_combined_loss.item()
#         batch_count += 1
#         batch_buffer = []

#         wandb.log({
#             "total_loss": total_combined_loss.item(),
#             "ce_loss": ce_loss.item(),
#             "aux_loss": aux_loss.item(),
#             "batch": batch_count
#         })

#         progress_bar.set_postfix({
#             "Total": f"{total_combined_loss.item():.4f}",
#             "CE": f"{ce_loss.item():.4f}",
#             "Aux": f"{aux_loss.item():.4f}"
#         })

#     if sample_count >= SUBSET_SIZE:
#         break

# avg_loss = total_loss / max(batch_count, 1)
# print(f"\nλ={lam} finished! Average Loss: {avg_loss:.4f}")

# wandb.finish()

# del model, optimizer, decoder_with_lora, base_decoder
# torch.cuda.empty_cache()
# gc.collect()
# print("GPU memory cleared!")

In [6]:
import gc

# ==========================================
# THE RANK (r) SWEEP: 3 EXPERIMENTS
# ==========================================

RANK_VALUES = [4, 8, 16]  # The 3 ranks we are testing
FIXED_LAMBDA = 1.0        # We found this was the best lambda!
SUBSET_SIZE = 5000        # Keep it small for the sweep (25 mins per run)

results_summary = {}

for r_val in RANK_VALUES:
    print(f"\n{'='*80}")
    print(f"  EXPERIMENT: LoRA Rank r = {r_val}")
    print(f"{'='*80}\n")

    # The standard math rule in LoRA is to keep alpha = 2 * r
    alpha_val = r_val * 2

    # --- 1. Start a fresh wandb run ---
    run = wandb.init(
        project="qwen-rank-sweep",
        name=f"r={r_val}",
        config={"lambda": FIXED_LAMBDA, "lora_r": r_val, "lora_alpha": alpha_val},
        reinit=True
    )

    # --- 2. Build a FRESH model from scratch ---
    base_decoder = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16, device_map={"": device}
    )

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM, 
        r=r_val,                # <--- Testing the new r here!
        lora_alpha=alpha_val,   # <--- Scaling alpha appropriately
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"]
    )
    decoder_with_lora = get_peft_model(base_decoder, lora_config)

    model = EndToEndInverter(
        paragraph_dim=PARAGRAPH_DIM,
        decoder_hidden_dim=base_decoder.config.hidden_size,
        prefix_len=PREFIX_LEN,
        decoder_model=decoder_with_lora
    )
    model = model.to(device)
    model = nn.DataParallel(model)
    model.train()

    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

    # --- 3. Stream 5,000 samples and train ---
    stream = load_dataset(
        "jg-eno/msmarco-v5.1-Qwen-Embeddings", split="train", streaming=True
    ).shuffle(seed=42)

    batch_buffer = []
    total_loss = 0
    batch_count = 0
    sample_count = 0

    progress_bar = tqdm(stream, desc=f"r={r_val}", total=SUBSET_SIZE)

    for sample in progress_bar:
        batch_buffer.append(sample)
        sample_count += 1

        if len(batch_buffer) == BATCH_SIZE:
            paragraph_embs, text_input_ids, attention_mask = prepare_batch(batch_buffer)
            paragraph_embs = paragraph_embs.to(device)
            text_input_ids = text_input_ids.to(device)
            attention_mask = attention_mask.to(device)

            optimizer.zero_grad()

            ce_loss, aux_loss = model(paragraph_embs, text_input_ids, attention_mask)
            ce_loss = ce_loss.mean()
            aux_loss = aux_loss.mean()

            # Using our winning lambda!
            total_combined_loss = ce_loss + FIXED_LAMBDA * aux_loss

            total_combined_loss.backward()
            optimizer.step()

            total_loss += total_combined_loss.item()
            batch_count += 1
            batch_buffer = []

            wandb.log({
                "total_loss": total_combined_loss.item(),
                "ce_loss": ce_loss.item(),
                "aux_loss": aux_loss.item(),
                "batch": batch_count
            })

            progress_bar.set_postfix({
                "Total": f"{total_combined_loss.item():.4f}",
                "CE": f"{ce_loss.item():.4f}"
            })

        if sample_count >= SUBSET_SIZE:
            break

    avg_loss = total_loss / max(batch_count, 1)
    results_summary[r_val] = avg_loss
    print(f"\nr={r_val} finished! Average Loss: {avg_loss:.4f}")

    wandb.finish()

    # --- 4. Clean up GPU memory before next experiment ---
    del model, optimizer, decoder_with_lora, base_decoder
    torch.cuda.empty_cache()
    gc.collect()

# ==========================================
# FINAL SUMMARY TABLE
# ==========================================
print("\n" + "=" * 50)
print("  RANK (r) SWEEP RESULTS")
print("=" * 50)
for r_val, avg in results_summary.items():
    print(f"  r = {r_val}  →  Average Loss = {avg:.4f}")
print("=" * 50)


  EXPERIMENT: LoRA Rank r = 4



wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

r=16:   0%|          | 0/5000 [00:00<?, ?it/s]


r=16 finished! Average Loss: 2.3738


aux_loss,██▇▆▆▅▄▅▃▃▂▁▂▂▂▂▁▂▁▂▂▁▁▂▂▁▁▁▂▂▁▁▂▁▁▂▂▂▂▁
batch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇█
ce_loss,▇▆▄▆▃▄▅▄▃▃▄▃█▄▄▂▄▄▄▂▃▃▃▄▂▄▂▄▄▅▆▃▄▁▃▃▄▃▃▂
total_loss,█▃▄▅▃▄▄▂▄▄▃▃▂▃▃▃▃▄▃▂▃▃▄▁▃▃▅▃▂▄▄▂▃▃▃▂▅▃▂▄
aux_loss,0.18573
batch,1250
ce_loss,1.80905
total_loss,1.99478



  RANK (r) SWEEP RESULTS
  r = 4  →  Average Loss = 2.4490
  r = 8  →  Average Loss = 2.3753
  r = 16  →  Average Loss = 2.3738
